# Generate portable synthetic CDC events

Offline path: use the included metadata-only fixture. Databricks path: use the active `spark` session. No credentials or source records are embedded.

## Prerequisites

Install `great-generator`. For SDK access outside a notebook, install `great-generator[databricks]` and use Databricks unified authentication. Metadata visibility and source object permissions are controlled by Unity Catalog.

In [ ]:
from pathlib import Path

from great_generator import StaticUnityCatalogClient

SOURCE_TABLE = "retail.sales.fact_orders"
FIXTURE = Path("examples/databricks/unity_catalog/mock_uc_metadata.json")
client = StaticUnityCatalogClient.from_path(FIXTURE)

# In a Databricks workspace, remove client=client and pass spark=spark instead.


## Generate synthetic data from metadata

In [ ]:
from great_generator import generate_uc_cdc

cdc = generate_uc_cdc(
    SOURCE_TABLE,
    rows=100,
    operations={"INSERT": 0.7, "UPDATE": 0.2, "DELETE": 0.1},
    seed=42,
    client=client,
)
cdc.head()


## Validate

In [ ]:
assert cdc["sequence_number"].is_monotonic_increasing
assert set(cdc["operation"]).issubset({"INSERT", "UPDATE", "DELETE"})
# Portable events are not represented as actual Delta Change Data Feed rows.


## Cleanup and limitations

These examples do not write tables. Remove only sandbox artifacts you explicitly create. Generated data is synthetic, not anonymized production data. Complex types and cyclic/self-referencing relationships require explicit modeling.